In [1]:
import numpy as np
import pandas as pd

# Step 1. Calcualte EMA Crossover
### $EMA(P_t\frac{1}{n_{k,s}}) - EMA(P_t\frac{1}{n_{k,l}})$
- x > 0: long
- x < 0: short

In [25]:
EMA_PAIRS = [(8, 24), (16, 48), (32, 96)] # (n_ks, n_kl)

def ema(x, n):
    return x.ewm(alpha=1/n, adjust=False).mean() # Exponentially Weighted Moving

def ema_crossovers(price):
    signals = {}
    
    for k, (n_s, n_l) in enumerate(EMA_PAIRS, start=1):
        
        short = ema(price, n_s)
        long = ema(price, n_l)
        
        signals[f"x{k}"] = short - long
        
    return signals

# Step 2. First Volatility Normalization
### $y_{k,t} = \frac{x_{k,t}}{\sigma_{63}(P_t)}$
- 63 days for conventional markets: 3 months of market arctivity
- 91 for crypto market: market never closes


In [22]:
def first_norm(price, x, crypto=True):
    
    window = 91 if crypto else 63
    
    price_vol = price.rolling(window).std()
    
    return x /price_vol

# Step 3. Normalize Entire Signal
### $z_{k,t} = \frac{y_{k,t}}{\sigma(y_k)}$ 
- 252 days for conventional markets
- 365 days for crypto market

In [4]:
def second_norm(y, crypto=True):
    
    window = 365 if crypto else 252
    
    rolling_vol = y.rolling(window).std()
    
    return y / rolling_vol

# Step 4. Nonlinear Response Function
###  $u(z) =\frac{ze^{-\frac{z^2_k}{4} }}{\sqrt{2}e^{-\frac{1}{2}}}~~$    $~-1\le u \le 1$

In [5]:
def u_func(z):
    den = np.sqrt(2) * np.exp(-0.5)
    
    return z * np.exp(-(z ** 2) / 4) / den

# Step 5. Create Combined Signal
### $\text{Signal}_t = \frac{1}{3}u_{1,t} + \frac{1}{3}u_{2,t} + \frac{1}{3}$

In [23]:
def momentum_signal(price, crytpo=True):
    
    xs = ema_crossovers(price)
    
    us = []
    
    for x in xs.values():
        
        y = first_norm(price, x, crypto=crytpo)
        
        z = second_norm(y, crypto=crytpo)
        
        u = u_func(z)
        
        us.append(u)
        
    return sum(us) / len(us)

# Portfolio Construction

## Strategy Returns with Signal Lag
### $\text{Signal}_{t-1}R_t$<br>

## Time-series Portfolio
### $w_{i,t}=\frac{\text{Signal}_{i,t}}{N}$

In [8]:
def ts_portfolio(signals, returns):
    
    n = signals.notna().sum(axis=1)
    
    weights = signals.div(n, axis=0)
    
    weights = weights.fillna(0)
    
    port_returns = (weights.shift(1) * returns).sum(axis=1, min_count=1)
    
    return port_returns, weights

## Cross-sectional Portfolio

In [9]:
def cs_weights(signals, n_l=3, n_s=3):
    
    weights = pd.DataFrame(0.0, index=signals.index, columns=signals.columns)
    
    total_positions = n_l + n_s
    position_size = 1 / total_positions
    
    for date, row in signals.iterrows():
        
        valid = row.dropna()
        if len(valid) < total_positions:
            continue
        
        l_assets = valid.nlargest(n_l).index
        s_assets = valid.nsmallest(n_s).index
        
        weights.loc[date, l_assets] = position_size
        weights.loc[date, s_assets] = -position_size
        
    return weights

def cs_portfolio(signals, returns):
    
    weights = cs_weights(signals)
    
    port_returns = (weights.shift(1) * returns).sum(axis=1, min_count=1)
    
    return port_returns, weights
    

# Rets and Metrics
## Data

In [10]:
from binance.client import Client as bnb_client
import os

### API Object

In [11]:
API_KEY = os.getenv('BINANCE_API_KEY')
API_SECRET = os.getenv('BINANCE_API_SECRET')

client = bnb_client(API_KEY, API_SECRET, tld='us')

In [12]:
univ = [ 'BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'BCHUSDT', 'AAVEUSDT', 'LTCUSDT', 
        'AVAXUSDT', 'LINKUSDT', 'ETCUSDT', 'XRPUSDT', 'NEOUSDT', 'XLMUSDT', 'ADAUSDT',
        'ATOMUSDT', 'NEARUSDT', 'FETUSDT', 'SUIUSDT', 'DASHUSDT', 'GRTUSDT', 'ICPUSDT',
        'DOTUSDT', 'UNIUSDT', 'VETUSDT', 'CHZUSDT', 'FILUSDT', 'THETAUSDT', 'DIAUSDT',
        'APTUSDT', 'ZENUSDT']

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def get_binance_px(symbol:str, freq:str, start_ts:str,end_ts:str) -> pd.DataFrame:
    
    data = client.get_historical_klines(symbol, freq, start_ts, end_ts)
    columns = ['open_time','open','high','low','close','volume','close_time','quote_volume',
    'num_trades','taker_base_volume','taker_quote_volume','ignore']

    data = pd.DataFrame(data, columns = columns)
    
    # Convert from POSIX timestamp (number of millisecond since jan 1, 1970)
    data['open_time'] = pd.to_datetime(data['open_time'], unit='ms')
    data['close_time'] = pd.to_datetime(data['close_time'], unit='ms')
    
    # enforce data types
    float_cols = ['open','high','low','close','volume',
                    'quote_volume','taker_base_volume','taker_quote_volume']
    
    data[float_cols] = data[float_cols].astype(float)
    
    return data

end = pd.to_datetime('2024-05-01')
start = end - pd.DateOffset(years=2)
start, end = start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')

freq = '1d'

def fetch_symbol(symbol):
    for attempt in range(3):
        try:
            data = get_binance_px(symbol, freq, start_ts=start, end_ts=end)
            return symbol, data.set_index('open_time')['close']
        except Exception as _:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                raise

px = {}
with ThreadPoolExecutor(max_workers=5) as executor: # network threading
    futures = {executor.submit(fetch_symbol, x): x for x in univ}
    for future in as_completed(futures):
        symbol, series = future.result()
        px[symbol] = series
    
px = pd.DataFrame(px).sort_index()
px.to_pickle('training_data.pk')


In [13]:
px = pd.read_pickle('training_data.pk')

## Rets

In [14]:

returns = px.pct_change(fill_method=None)
print("Num of observations: ", returns.shape[0])
print("Num of assets: ", returns.shape[1])
print(f"Time period start: {returns.index[0]}      Time period end: {returns.index[-1]}")
returns.head()

Num of observations:  732
Num of assets:  30
Time period start: 2022-05-01 00:00:00      Time period end: 2024-05-01 00:00:00


,BNBUSDT,BCHUSDT,BTCUSDT,SOLUSDT,ETHUSDT,AVAXUSDT,LINKUSDT,ETCUSDT,AAVEUSDT,LTCUSDT,...,ICPUSDT,DOTUSDT,CHZUSDT,UNIUSDT,VETUSDT,THETAUSDT,FILUSDT,APTUSDT,ZENUSDT,DIAUSDT
open_time,,,,,,,,,,,,,,,,,,,,,
2022-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-05-02,-0.000792,0.003664,0.000829,-0.024358,0.011203,0.027344,-0.014172,-0.004888,-0.016248,0.011881,...,NaN,-0.024104,-0.019759,-0.027463,-0.019339,NaN,-0.021941,NaN,-0.021793,NaN
2022-05-03,-0.014352,-0.020778,-0.020052,-0.019328,-0.026262,-0.013887,-0.001797,-0.028944,-0.015402,-0.010448,...,NaN,-0.016021,-0.003359,-0.007909,-0.014727,NaN,-0.013596,NaN,0.000595,NaN
2022-05-04,0.049533,0.076669,0.051020,0.080773,0.057475,0.126907,0.097210,0.151019,0.130309,0.068577,...,NaN,0.105834,0.101124,0.158860,0.153387,NaN,0.082012,NaN,0.113178,NaN
2022-05-05,-0.060372,-0.080698,-0.078475,-0.088462,-0.066070,-0.122136,-0.105824,-0.077831,-0.116851,-0.089301,...,NaN,-0.104294,-0.100000,-0.081772,-0.118675,NaN,-0.112102,NaN,-0.078494,NaN


In [26]:
signals = momentum_signal(px, crytpo=True)

In [27]:
ts_rets, ts_weights = ts_portfolio(signals, returns)

cs_rets, cs_wgts = cs_portfolio(signals, returns)

## Metrics
1. Performance
2. Transaction Costs

In [17]:
def metrics(rets, crypto=True):
    
    periods = 365 if crypto else 252
    
    ann_rets = rets.mean() * periods
    ann_vol = rets.std() * np.sqrt(periods)
    sharpe = ann_rets / ann_vol
    
    return {"Annual Return": ann_rets,
            "Annual Volatility": ann_vol,
            "Sharpe": sharpe}

In [18]:
def tcosts(rets, weights, cost_bps=20):
    
    turnover = weights.diff().abs().sum(axis=1)
    cost = turnover * cost_bps / 10_000
    net_rets = rets - cost.shift(1).fillna(0)
    
    return net_rets, turnover

### Before Tcosts

In [19]:
from pprint import pprint as pp

In [28]:
pp(metrics(ts_rets))

{'Annual Return': 0.2368477941703544,
 'Annual Volatility': 0.26713756940255357,
 'Sharpe': 0.886613570304089}


In [29]:
pp(metrics(cs_rets))

{'Annual Return': -0.07383435493638621,
 'Annual Volatility': 0.16294535748589986,
 'Sharpe': -0.4531234033027011}


## Backtest